# Bistable Mass-in-Mass Simulation

### Imports


In [ ]:
import jax.numpy as jnp
from jax import grad, jit, vmap
from jax.experimental.ode import odeint
from jax import config
config.update("jax_enable_x64", True)  # enable float64 type
from pathlib import Path
from matplotlib import animation
from matplotlib import cm, colors
import matplotlib.animation as animation
from matplotlib.collections import (LineCollection, PolyCollection)
import matplotlib
import matplotlib.pyplot as plt
from colors import color_scheme
colors_lst, red, custom_cmap = color_scheme()
from jax import grad, jit

In [ ]:
def pixel_per_meter(raw_data, multiple_units=False):
    ref_length = 86*10**(-3) # [m]

    if multiple_units:
        n_pixels_ = jnp.array([jnp.abs(raw_data[0, 0] - raw_data[1, 0]), jnp.abs(raw_data[0, 0] - raw_data[1, 0]), jnp.abs(raw_data[2, 0] - raw_data[3, 0]), jnp.abs(raw_data[4, 0] - raw_data[5, 0]), jnp.abs(raw_data[6, 0] - raw_data[7, 0]), jnp.abs(raw_data[8, 0] - raw_data[9, 0]), jnp.abs(raw_data[10, 0] - raw_data[11, 0])])
        n_pixels = jnp.ones([(n_pixels_.shape[0])*2-1, 2])
        for i in jnp.arange(n_pixels_.shape[0]):
            n_pixels = n_pixels.at[2*i, :].set(n_pixels_[i])
            n_pixels = n_pixels.at[2*i+1, :].set(n_pixels_[i])
    else:    
        if len(raw_data.shape) > 2:
            n_pixels = jnp.abs(raw_data[0, 0, 0] - raw_data[0, 1, 0])
        else:
            n_pixels = jnp.abs(raw_data[0, 0] - raw_data[1, 0])
    return n_pixels/ref_length  

### Define energy potential

In [ ]:
# Parameters fitted to experimental data
k2 = 125
k4 = 309828
k3 = 3*jnp.sqrt(k2*k4/2) # ensures a symmetric potential well
k_fit4 = jnp.array([k2, k4])
k_truss = 1428

# Import geometric data (L, theta0)
if True:
    ppms = pixel_per_meter(jnp.load(f'Audrey_exp_data/px_mm_conversion.npy')[:, :], multiple_units=False) 
    data_set_L_values = ((jnp.load(f'Audrey_exp_data/data_for_L_values.npy'))/ppms)[:, 1]
    data_set_theta_values = ((jnp.load(f'Audrey_exp_data/data_for_theta_values.npy'))/ppms)
    data_set_b_value = ((jnp.load(f'Audrey_exp_data/data_for_b_value.npy'))/ppms)

    b = jnp.sqrt( (data_set_b_value[0, 0] - data_set_b_value[1, 0])**2 + (data_set_b_value[0, 1] - data_set_b_value[1, 1])**2 )

    L = jnp.abs(jnp.diff(data_set_L_values)[::2])/2

    theta0 = jnp.zeros((data_set_theta_values.shape[0])//2)
    for i in jnp.arange(data_set_theta_values.shape[0]//2):
        p1 = data_set_theta_values[2*i, :]
        p2 = data_set_theta_values[2*i+1, :]

        side_adj = jnp.abs(p1[1] - p2[1])
        side_opp = jnp.abs(p1[0] - p2[0])

        theta0 = theta0.at[i].set((jnp.arctan(side_opp/side_adj)))


def potential_order4(ks, x):
    'ks = k_fit4'
    'x = d2-d1'
    k2 = ks[0]
    k4 = ks[1]

    k3 = 3*jnp.sqrt(k2*k4/2)

    return k2/2*(x)**2 - k3/3*(x)**3 + k4/4*(x)**4


def potential_order4_summed(ks, x):
    'ks = k_fit4'
    'x = d2-d1'
    k2 = ks[0]
    k4 = ks[1]

    k3 = 3*jnp.sqrt(k2*k4/2)

    return jnp.sum(k2/2*(x)**2 - k3/3*(x)**3 + k4/4*(x)**4, axis=-1)


def bistable_potential(k, x):
    # k is the effective spring constant accounting for both axial spring stiffnesses (e.g., k = k1 + k2 = 2k1 where k1 is the axial stiffness of one of the two springs forming the truss)
    l0 = L[0]/jnp.cos(theta0[0]) - 2*b
    a0 = (l0+2*b)*jnp.sin(theta0[0]) 
    Ex = (k * (L[0]/jnp.cos(theta0[0]) - jnp.sqrt((a0-x)**2 + L[0]**2))**2) / 2
    Ex = Ex# - jnp.min(Ex)
    return Ex


def bistable_potential_summed(k, x):
    l0 = L[0]/jnp.cos(theta0[0]) - 2*b
    a0 = (l0+2*b)*jnp.sin(theta0[0]) 
    Ex = (k * (L[0]/jnp.cos(theta0[0]) - jnp.sqrt((a0-x)**2 + L[0]**2))**2) / 2
    Ex = Ex# - jnp.min(Ex)
    return jnp.sum(Ex, axis=0)

x_eq2 = k3/(2*k4) + jnp.sqrt( (k3/(2*k4))**2 - k2/k4 )
x_eqs_O4 = jnp.array([0, x_eq2]) # equilibrium positions (i.e., displacement value at energy minimas)

d1 = 0 # displacement of outer mass
d2 = jnp.linspace(-0.005, 0.031, 1000) # displacement of inner mass
fitted_energy4 = potential_order4(k_fit4, d2)
fitted_energy_VonMisses = bistable_potential(2*k_truss, d2)

# Visualize the energy landscape
plt.figure(figsize=[4,3])
# plt.plot(d2*10**3, fitted_energy4*10**3, 'k')
plt.plot(d2*10**3, fitted_energy_VonMisses*10**3, 'r')
plt.xlabel('Displacement (mm)')
plt.ylabel('Energy (mJ)')
plt.title('Bistable Energy Landscape')
plt.tight_layout()

In [ ]:
b

### Set up the simulation

In [ ]:
# Define function to return local and global states based on constraints
def get_local_global_states(n_dof, m1_state0, m2_state0, m1_constrained_unit_ids, m2_constrained_unit_ids):

    # Define global state indices as <u_0, v_0, u_1, v_1, u_2, v_2, .... u_n, v_n>
    state0_global = jnp.zeros([2, n_dof])
    state0_global = state0_global.at[:, 0::2].set(m1_state0)
    state0_global = state0_global.at[:, 1::2].set(m2_state0)

    if len(m1_constrained_unit_ids)>0:
        m1_ad_free_dof_ids = jnp.delete(jnp.arange(0, n_dof, 2), m1_constrained_unit_ids)
    else:
        m1_ad_free_dof_ids = jnp.arange(0, n_dof, 2)
    if len(m2_constrained_unit_ids)>0:
        m2_ad_free_dof_ids = jnp.delete(jnp.arange(1, n_dof, 2), m2_constrained_unit_ids)
    else:
        m2_ad_free_dof_ids = jnp.arange(1, n_dof, 2)


    state0_local = jnp.zeros([2, len(m1_ad_free_dof_ids)+len(m2_ad_free_dof_ids)])
    state0_local = state0_local.at[:, 0::2].set(state0_global[:, m1_ad_free_dof_ids])
    state0_local = state0_local.at[:, 1::2].set(state0_global[:, m2_ad_free_dof_ids])

    return state0_local, state0_global

# Define function to return local and global states based on constraints and phase
def get_current_local_global_states(phase_description, n_units, x_eq2s, m1_state0, m1_constrained_unit_ids, m2_constrained_unit_ids):
    '''
    gives updated local and global state with updated phase description
    '''
    n_dof = 2*n_units
    p2_units = []
    for i, unit_phase in enumerate(phase_description):
        if unit_phase == '1':
            p2_units.append(i)

    m2_state0_p2 = jnp.zeros([2, n_units]) # displacement, velocity starting in phase 2
    if len(p2_units)>0:
        p2_unit_ids = jnp.asarray(p2_units) + 1
        m2_state0_p2 = m2_state0_p2.at[0, p2_unit_ids].set(x_eq2s)
    state0_local_, state0_global_ = get_local_global_states(n_dof, m1_state0, m2_state0_p2, m1_constrained_unit_ids, m2_constrained_unit_ids)

    return state0_local_, state0_global_

def get_system_state(vn_minus_un: jnp.ndarray, threshold: float = 15e-3) -> str:
    """Return the three-unit state as bits based on ``v_n - u_n``.

    A unit is represented by ``'1'`` when its relative displacement exceeds
    ``threshold`` and by ``'0'`` otherwise. The default threshold is 15 mm.
    """
    relative_displacements = jnp.asarray(vn_minus_un).reshape(-1)
    if relative_displacements.size != 3:
        raise ValueError('vn_minus_un must contain exactly three unit displacements.')

    return ''.join('1' if float(value) > threshold else '0'
                   for value in relative_displacements)

def state_to_number(state: str) -> int:
    """Convert a three-bit system state to its integer value."""
    if len(state) != 3 or any(bit not in '01' for bit in state):
        raise ValueError("state must be a three-character binary string, e.g. '001'.")

    return int(state, 2)

def get_model_info(potential_model_name, n_units):

    'potential_model_name: (str) --> "4th Order" or "Trusses"'
    'n_units: (int) --> number of units in the simulation (including fixed unit at beginning and end)'

    if potential_model_name == '4th Order':
          # Potential function
          potential_fn = potential_order4
          potential_summed = potential_order4_summed

          # Potential parameters
          bistable_potential_params = jnp.zeros([n_units, k_fit4.shape[0]])
          bistable_potential_params = bistable_potential_params.at[1:-1,:].set(k_fit4*jnp.ones_like(bistable_potential_params.shape[0]))

          # Equilibrium points
          equilibrium1 = x_eqs_O4[0]
          equilibrium2 = x_eqs_O4[1]

    elif potential_model_name == 'Trusses':
          # Von Mises potential; k_truss is the stiffness of one arm.
          potential_fn = bistable_potential
          potential_summed = bistable_potential_summed

          # bistable_potential includes the 1/2 factor, so pass the
          # combined stiffness of the two identical truss arms.
          effective_k_truss = 2*k_truss
          bistable_potential_params = jnp.zeros([n_units, 1])
          bistable_potential_params = bistable_potential_params.at[1:-1, 0].set(effective_k_truss)

          # The two symmetric minima occur at x=0 and x=2*a0.
          l0 = L[0]/jnp.cos(theta0[0]) - 2*b
          a0 = (l0 + 2*b)*jnp.sin(theta0[0])
          equilibrium1 = 0.0
          equilibrium2 = 2*a0

    else:
          raise ValueError(f'Unknown potential model: {potential_model_name}')
    
    return potential_fn, potential_summed, bistable_potential_params, equilibrium1, equilibrium2

def get_free_dof_parameters(m1s, m2s, c1s, c2s, mu_ks, stiffness_vals, m1_constrained_ids, m2_constrained_ids):

    if len(m1_constrained_ids)>0:
        m1s_free = jnp.delete(m1s, m1_constrained_ids)
        c1s_free = jnp.delete(c1s, m1_constrained_ids)
        mu_ks_free = jnp.delete(mu_ks, m1_constrained_ids)
    else:
        m1s_free = m1s
        c1s_free = c1s
        mu_ks_free = mu_ks
    if len(m2_constrained_ids)>0:
        m2s_free = jnp.delete(m2s, m2_constrained_ids)
        c2s_free = jnp.delete(c2s, m2_constrained_ids)
        stiffness_vals_free = jnp.delete(stiffness_vals, m2_constrained_ids, axis=0)
    else:
        m2s_free = m2s
        c2s_free = c2s
        stiffness_vals_free = stiffness_vals

    return m1s_free, m2s_free, c1s_free, c2s_free, mu_ks_free, stiffness_vals_free

# Define plotting function
def plot_response(num_solution, timepoints, alpha=1, cmap_temporal='Dark2', sup_title=None, figsize=[8, 5]):
    'Plot spatiotemporal plots and temporal signals of un and vn-un of simulations'
    def cmap_to_color_list(cmap_name, n_discrete_regions):
        cmap = cm.get_cmap(cmap_name, n_discrete_regions)

        color_list = []
        for i in range(cmap.N):
            rgba = cmap(i)
            color_list.append(matplotlib.colors.rgb2hex(rgba))
        
        return color_list

    n_units = num_solution.shape[2] - 2

    if cmap_temporal != 'Dark2':
        colors = cmap_to_color_list(cmap_temporal, n_units)
    else:
        colors = cmap_to_color_list(cmap_temporal, 8)

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=figsize)

    orig_map=plt.cm.get_cmap('plasma')
    rev_cmap = orig_map.reversed()

    data_set = num_solution*10**(3)

    un_disps = data_set[:, 0, 1:-1]
    vn_disps = data_set[:, 2, 1:-1]

    t_end = float(jnp.max(timepoints))
    time_edges = jnp.linspace(0, t_end, un_disps.shape[0] + 1)
    unit_edges = jnp.arange(un_disps.shape[1] + 1)

    p1 = ax1.pcolor(time_edges, unit_edges, un_disps.T, cmap=rev_cmap)
    p2 = ax2.pcolor(time_edges, unit_edges, (vn_disps - un_disps).T, cmap=rev_cmap)

    plt.colorbar(p1, ax=ax1, pad=0.01, label='$u_n$ (mm)')
    plt.colorbar(p2, ax=ax2, pad=0.01, label='$v_n-u_n$ (mm)')

    ax1.set_title('Displacement, $u_n$', fontsize=12)
    ax2.set_title('Displacement, $v_n - u_n$', fontsize=12)

    ax1.set_yticks(jnp.arange(0.5, 0.5+un_disps.shape[1]))
    ax1.set_yticklabels(jnp.arange(un_disps.shape[1]), fontsize=12)
    ax2.set_yticks(jnp.arange(0.5, 0.5+un_disps.shape[1]))
    ax2.set_yticklabels(jnp.arange(un_disps.shape[1]), fontsize=12)

    ax1.set_xlim([0, t_end])
    ax2.set_xlim([0, t_end])


    ax1.set_xlabel('Time (s)', fontsize=12)
    ax2.set_xlabel('Time (s)', fontsize=12)

    ax1.set_ylabel('$n$', fontsize=12)
    ax2.set_ylabel('$n$', fontsize=12)

    for i in jnp.arange(n_units):
        ax3.plot(timepoints, un_disps[:, i], color=colors[i], alpha=alpha, label=f'$u_{i}$')

        ax4.plot(timepoints, (vn_disps[:, i]-un_disps[:, i]), color=colors[i], alpha=alpha, label=f'$v_{i}-u_{i}$')
    
    ax3.legend(fontsize=6, loc='upper right')
    ax4.legend(fontsize=6, loc='upper right')

    ax3.set_xlim([0, t_end])
    ax4.set_xlim([0, t_end])
    
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('Displacement (mm)')
    ax3.set_title('$u_n$ (mm)')
    ax4.set_xlabel('Time (s)')
    ax4.set_ylabel('Displacement (mm)')
    ax4.set_title('$v_n-u_n$ (mm)')

    if sup_title is not None:
        plt.suptitle(f'{sup_title}')
        
    plt.tight_layout()



In [ ]:
# Define mass profile in the chain
n_units = 3
n_units+=2 # account for constrained ends

# Define Physical parameters
# a = 44.8*10**(-3) # unit cell length, [m]
m1 = 70*10**(-3) # m_out [kg]
m2 = 30*10**(-3) # m_in [kg]
k1 = 125 # [N/m]
c1 = 0.22 # c_out [kg/s]
c2 = 0.1 # c_in [kg/s]
mu_k = 0.0675 # coefficient of kinetic friction
beta = 100 # smoothing factor for friction force [s/m]
# potential_model_name = '4th Order' # 4th order polynomial
potential_model_name = 'Trusses' # all identical trusses from Audrey data of previous paper

potential_fn, potential_summed,bistable_potential_params, x_eq1s, x_eq2s = get_model_info(potential_model_name, n_units)
stiffness_vals = jnp.concatenate([k1*jnp.ones([n_units,1]), bistable_potential_params], axis=-1)

# Define state variables
m1_state0 = jnp.zeros([2, n_units]) # displacement, velocity
m2_state0 = jnp.zeros([2, n_units]) # displacement, velocity starting in phase 1
m1_state0 = m1_state0.at[0, :].set(0.00000000001)

n_dof = 2*n_units # Number of degrees of freedom. Each unit has 2 dof

# Define constrained unit ids
m1_constrained_unit_ids = jnp.array([0, -1]) # First unit is driven, last unit is fixed
m2_constrained_unit_ids = jnp.array([0, -1]) # Inner mass of unit is driven, inner mass of last unit is fixed
constrained_unit_ids = (m1_constrained_unit_ids, m2_constrained_unit_ids)

# Define zero displacement for constrained units
def constsignal(t):
     return 0
def dconstsignal(t):
     return 0

# Reshape the local state variable to global state variable
def reshape_local_to_global(free_dof_solution, t, timepoints, impulse_fn_data, amplitude_impulse_scaling=1.):
     global_solution = jnp.zeros([timepoints.shape[0], 4, n_units])

     def impulse_fn(t_):
          return jnp.interp(t_, timepoints, amplitude_impulse_scaling*impulse_fn_data)

     def dimpulse_fn(t_):
          ds = grad(impulse_fn)
          return ds(t_)

     global_solution = global_solution.at[:, 0, 0].set(impulse_fn(t)) # applied displacement
     global_solution = global_solution.at[:, 1, 0].set(vmap(dimpulse_fn)(t)) # applied velocity
     global_solution = global_solution.at[:, 2, 0].set(impulse_fn(t)) # applied displacement
     global_solution = global_solution.at[:, 3, 0].set(vmap(dimpulse_fn)(t)) # applied velocity

     # displacement values
     global_solution = global_solution.at[:, 0, 1:-1].set(free_dof_solution[:, 0, 0::2]) # free m1 dof disp solution
     global_solution = global_solution.at[:, 2, 1:-1].set(free_dof_solution[:, 0, 1::2]) # free m2 dof disp solution
     # velocity values
     global_solution = global_solution.at[:, 1, 1:-1].set(free_dof_solution[:, 1, 0::2]) # free m1 dof vel solution
     global_solution = global_solution.at[:, 3, 1:-1].set(free_dof_solution[:, 1, 1::2]) # free m2 dof disp solution
     
     return global_solution

# Define potential energy of the constrained spring-mass system
def potential_energy_constrained_ends(free_dof_displacement, t, stiffness_vals, impulse_fn):
     "free_dof_displacement: <1xn_free_dof> assumes u_0, u_-1, v_-1 are all constrained --> jnp.array([u_1, v_1, u_2, v_2, u_3, v_3, ...., u_-2, v_-2])"
     "t: time"
     "stiffness_vals: jnp.array([k1, k2, k3, k4,...]) used to calculate the coupling potential"
     "impulse_fn: imposed displacement (impulse1(t), impulse2(t), gaussian_impulse(t), etc.)"

     displacement1 = jnp.zeros(n_units)
     displacement2 = jnp.zeros(n_units)

     displacement1 = displacement1.at[0].set(impulse_fn(t)) # impose applied displacement
     displacement2 = displacement2.at[0].set(impulse_fn(t)) # impose applied displacement

     displacement1 = displacement1.at[-1].set(constsignal(t)) # impose fixed end
     displacement2 = displacement2.at[-1].set(constsignal(t)) # impose fixed end

     displacement1 = displacement1.at[1:-1].set(free_dof_displacement[0::2]) # set the free dofs
     displacement2 = displacement2.at[1:-1].set(free_dof_displacement[1::2]) # set the free dofs

     coupling_stiffness = stiffness_vals[:, 0]
     potential_params = stiffness_vals[:, 1:].T
     potential_energy_constr = 0.5 * jnp.sum(coupling_stiffness[0] * jnp.diff(displacement1)**2) + jnp.sum(potential_fn(potential_params, displacement2-displacement1))

     return potential_energy_constr

# Calculate the force as the gradient of the potential energy
force_constrained_ends = grad(lambda free_dof_displacement, t, stiffness_vals, impulse_fn: -potential_energy_constrained_ends(free_dof_displacement, t, stiffness_vals, impulse_fn), argnums=0)

# potential_energy_fn = potential_energy_constrained_ends
force_fn = force_constrained_ends

# Define RHS
@jit
def rhs(state, t, timepoints, m1s, m2s, stiffness_vals, c1s, c2s, mu_ks, beta, impulse_fn_data):
     "state: local state0 < 2 x n_free_dof >"
     "t: time"
     "timepoints: discrete timepoints"
     "m1: mass 1 (n_units, 1)"
     "m2: mass 2 (n_units, 1)"
     "stiffness_vals: (n_units, len(bistable_potential_params)+1) to be passed to force_constr..."
     "c1s: c1 damping (n_units, 1)"
     "cc2: c2 damping (n_units, 1)"
     "mu_k: friction (n_units, 1)"
     "impulse_fn_data: impulse_fn(timepoints)"
     
     displacement, velocity = state
     
     def impulse_fn(t_):
          return jnp.interp(t_, timepoints, impulse_fn_data)
     
     mass_vals = jnp.ones_like(state[0])
     mass_vals = mass_vals.at[0::2].set(m1s)
     mass_vals = mass_vals.at[1::2].set(m2s) # array of m2, m1, m2, m1, m2

     # damping_un = c1s*(velocity[0::2]) + (mu_ks)*9.81*m1s*jnp.sign(velocity[0::2])
     damping_un = c1s*(velocity[0::2]) + mu_ks*9.81*m1s*jnp.tanh(velocity[0::2]*beta) - c2s*(velocity[1::2] - velocity[0::2])  #(mu_ks)*9.81*m1s*jnp.sign(velocity[0::2])
     damping_vn = jnp.zeros_like(velocity[1::2])
     damping_vn = damping_vn.at[:].set(c2s*(velocity[1::2] - velocity[0::2]))

     damping = jnp.zeros_like(velocity)
     damping = damping.at[0::2].set(damping_un)
     damping = damping.at[1::2].set(damping_vn)

     acceleration = (force_fn(displacement, t, stiffness_vals, impulse_fn) - damping)/mass_vals

     return jnp.array([velocity, acceleration])


def twogaussian_impulse(t, A, t_s, w, t_s2_factor=1.5):
    """
    t: timepoints (array)
    A: amplitude [m] (float)
    t_s: time_shift (float)
    w: width of pulse (float)
    t_s2_factor: scaling factor for impulse #2. defaults to 1.5 (float)
    """

    return A*jnp.exp(-(t-t_s)**2 / w**2)+A*jnp.exp(-(t-t_s2_factor*t_s)**2 / w**2)


def gaussian_impulse(t, A, t_s, w=None, f=None):
    """
    t: timepoints (array)
    A: amplitude [m] (float)
    t_s: time_shift (float)
    w: width of pulse (float)
    f: frequency of pulse (float)"""
    if f is not None:
        w = 1/(2*2*jnp.sqrt(2*jnp.log(2))*f)  # Width [s], used by gaussian
    elif w is None:
        raise ValueError("Either width 'w' or frequency 'f' must be provided.")
    return A*jnp.exp(-(t-t_s)**2 / w**2)


def single_sine_cycle(t: jnp.ndarray, A: float, t_s: float, f: float) -> jnp.ndarray:
    """Return one sinusoidal displacement cycle, followed by zero displacement.

    Parameters
    ----------
    t : jnp.ndarray
        Time points [s].
    A : float
        Peak displacement amplitude [m].
    f : float
        Frequency [Hz]; must be positive.
    t_s : float, optional
        Cycle start-time offset [s].
    """
    if f <= 0:
        raise ValueError('Frequency f must be positive.')

    phase = f*(t-t_s)
    return jnp.where((phase >= 0) & (phase < 1), A*jnp.sin(2*jnp.pi*phase), 0.0)

In [ ]:
# Define arrays of variables to pass into rhs function. Currently, all unit cells are identical.
m1s = m1*jnp.ones([n_units])
m2s = m2*jnp.ones([n_units])
c1s = c1*jnp.ones([n_units])
mu_ks = mu_k*jnp.ones([n_units])
c2s = c2*jnp.ones([n_units])

m1s_free, m2s_free, c1s_free, c2s_free, mu_ks_free, stiffness_vals_free = get_free_dof_parameters(m1s, m2s, c1s, c2s, mu_ks, stiffness_vals, m1_constrained_unit_ids, m2_constrained_unit_ids)

### Look at output of simulation for one impulse

In [ ]:
# Compare the same pulse for two initial phases
initial_phases = ('100', '110')

# Input parameters
input_type = 'single_sine_cycle' # Options: 'single_sine_cycle' or 'gaussian'
# input_type = 'gaussian' # Options: 'single_sine_cycle' or 'gaussian'
# A = 22*10**(-3) # Peak amplitude [m]
A = 20*10**(-3) # Peak amplitude [m]
f = 2.0 # Frequency [Hz], used by single_sine_cycle
w = 1/(2*2*jnp.sqrt(2*jnp.log(2))*f)  # Width [s], used by gaussian
# w = 0.005 # Width [s], used by gaussian
t_s = 0.075 # Start-time offset [s] for both input types
T = jnp.linspace(0, 4.0, 600) # Time (s)

if input_type == 'single_sine_cycle':
    impulse_to_model = single_sine_cycle
    impulse_data = impulse_to_model(T, A, t_s, f=f)
elif input_type == 'gaussian':
    impulse_to_model = gaussian_impulse
    # impulse_data = impulse_to_model(T, A, t_s, w)
    impulse_data = impulse_to_model(T, A, t_s, f=f)
else:
    raise ValueError(f'Unknown input_type: {input_type}')

# Integrate the same pulse from each initial phase
solutions_by_phase = {}
for initial_phase in initial_phases:
    state0_local_, _ = get_current_local_global_states(initial_phase, n_units, x_eq2s, m1_state0, m1_constrained_unit_ids, m2_constrained_unit_ids)
    sol_free = odeint(rhs, state0_local_, T, T, m1s_free, m2s_free, stiffness_vals, c1s_free, c2s_free, mu_ks_free, beta, impulse_data)
    solutions_by_phase[initial_phase] = reshape_local_to_global(sol_free, T, T, impulse_data)

# Plot the impulse
plt.figure(figsize=[4,2])
plt.plot(T, impulse_data*10**3, 'k')
plt.xlabel('Time (s)', fontsize=12)
plt.ylabel('$\\bar{u}(t)$ (mm)', fontsize=12)
plt.grid(False)
plt.xlim([0, max(T)])
plt.title('Applied Displacement')
plt.tight_layout()

# Keep one four-subplot response figure for each initial phase
for initial_phase, sol_global in solutions_by_phase.items():
    plot_response(sol_global, T, figsize=[8,5], sup_title=f'Initial phase {initial_phase}')

# Compare the endpoint spring forces and their difference between initial phases
if len(initial_phases) != 2:
    raise ValueError('The force-difference plot requires exactly two initial phases.')

final_spring_id = next(iter(solutions_by_phase.values())).shape[2] - 1
forces_by_phase = {}
fig, axes = plt.subplots(3, 1, figsize=[4, 6], sharex=True)
ax_first, ax_final, ax_difference = axes

for phase_id, (initial_phase, sol_global) in enumerate(solutions_by_phase.items()):
    first_spring_force = k1 * (sol_global[:, 0, 0] - sol_global[:, 0, 1])
    final_spring_force = k1 * (sol_global[:, 0, -2] - sol_global[:, 0, -1])
    forces_by_phase[initial_phase] = {
        'first': first_spring_force,
        'final': final_spring_force,
    }
    phase_color = colors_lst[phase_id]
    ax_first.plot(T, first_spring_force, color=phase_color, label=f'Initial {initial_phase}')
    ax_final.plot(T, final_spring_force, color=phase_color, label=f'Initial {initial_phase}')

def detect_force_arrival(timepoints: jnp.ndarray, force: jnp.ndarray, start_time: float,
                         threshold_fraction: float = 0.05) -> float:
    """Return the first post-input time at which force reaches a fraction of its peak."""
    post_input = timepoints >= start_time
    peak_force = jnp.max(jnp.where(post_input, jnp.abs(force), 0.0))
    if float(peak_force) == 0.0:
        raise ValueError('Cannot detect arrival in a zero force signal.')
    has_arrived = post_input & (jnp.abs(force) >= threshold_fraction * peak_force)
    return float(timepoints[int(jnp.argmax(has_arrived))])

phase_a, phase_b = initial_phases
reference_force = forces_by_phase[phase_a]
first_arrival_time = detect_force_arrival(T, reference_force['first'], t_s)
final_arrival_time = detect_force_arrival(T, reference_force['final'], t_s)
force_delay = final_arrival_time - first_arrival_time
if force_delay < 0:
    raise ValueError('Detected final-spring arrival before first-spring arrival.')

delta_first = forces_by_phase[phase_a]['first'] - forces_by_phase[phase_b]['first']
delta_final = forces_by_phase[phase_a]['final'] - forces_by_phase[phase_b]['final']
delta_final_aligned = jnp.interp(
    T + force_delay, T, delta_final, left=jnp.nan, right=jnp.nan
)
ax_difference.plot(T, delta_first, color=colors_lst[0], label=fr'$\Delta F_1(t)$')
ax_difference.plot(T, delta_final_aligned, color=red, linestyle='--',
                   label=fr'$\Delta F_{{{final_spring_id}}}(t+\tau)$')
ax_difference.axhline(0, color='k', linewidth=0.8, alpha=0.35)
ax_first.axvline(first_arrival_time, color='k', linestyle=':', alpha=0.4)
ax_final.axvline(final_arrival_time, color='k', linestyle=':', alpha=0.4)

ax_first.set_ylabel(fr'$F_1$ (N)', fontsize=12)
ax_final.set_ylabel(fr'$F_{{{final_spring_id}}}$ (N)', fontsize=12)
ax_difference.set_ylabel(fr'$F^{{{phase_a}}}-F^{{{phase_b}}}$ (N)', fontsize=12)
ax_difference.set_xlabel('Time (s)', fontsize=12)
ax_first.set_title(f"Endpoint Forces for Initial Phases {phase_a} and {phase_b}")
ax_difference.set_title(fr'Arrival aligned using $\tau={force_delay*1e3:.1f}$ ms from {phase_a}')
for ax in axes:
    ax.grid(False)
    ax.set_xlim([0, max(T)])
    ax.legend()
plt.tight_layout()
print(f'Force-arrival delay for initial phase {phase_a}: {force_delay*1e3:.1f} ms')

# Read the final three-unit state of each simulation
for initial_phase, sol_global in solutions_by_phase.items():
    final_vn_minus_un = sol_global[-1, 2, 1:-1] - sol_global[-1, 0, 1:-1]
    final_state = get_system_state(final_vn_minus_un)
    final_state_number = state_to_number(final_state)
    print(f'Initial {initial_phase} -> final state: {final_state} ({final_state_number})')

In [ ]:
import sys
sys.exit("program complete, Exiting kernel.")

### Quick amplitude-frequency parameter study

In [ ]:
initial_phase = '110' # Initial phase of the structure represented as bits
input_type = 'single_sine_cycle' # Options: 'single_sine_cycle' or 'gaussian'
A = 22*10**(-3) # Peak amplitude [m]
f = 20 # Frequency [Hz], used by single_sine_cycle
w = 1/(2*2*jnp.sqrt(2*jnp.log(2))*f)  # Width [s], used by gaussian

# Get the local and global states based on the initial_phase
state0_local_, state0_global_ = get_current_local_global_states(initial_phase, n_units, x_eq2s, m1_state0, m1_constrained_unit_ids, m2_constrained_unit_ids)


if input_type == 'single_sine_cycle':
    impulse_to_model = single_sine_cycle
elif input_type == 'gaussian':
    impulse_to_model = gaussian_impulse
else:
    raise ValueError(f'Unknown input_type: {input_type}')

N_axis = 40
A_small_mm, A_large_mm = 5, 25
A_values_mm = jnp.concatenate([
    jnp.linspace(-A_large_mm, -A_small_mm, N_axis//2),
    jnp.linspace(A_small_mm, A_large_mm, N_axis//2),
])
f_values_hz = jnp.linspace(8, 22, N_axis)
study_T = jnp.linspace(0, 1.0, 300)

def simulate_final_state(A_mm: float, f_hz: float) -> str:
    """Simulate one sine input and return its final three-bit state."""
    input_data = impulse_to_model(study_T, A_mm*1e-3, t_s, f=f_hz)
    solution = odeint(rhs, state0_local_, study_T, study_T, m1s_free, m2s_free,
                      stiffness_vals, c1s_free, c2s_free, mu_ks_free, beta, input_data)
    return get_system_state(solution[-1, 0, 1::2] - solution[-1, 0, 0::2])

parameter_states = []
for A_mm in A_values_mm:
    for f_hz in f_values_hz:
        final_state = simulate_final_state(A_mm, f_hz)
        parameter_states.append((float(A_mm), float(f_hz), final_state))

state_number_grid = jnp.array([state_to_number(row[2]) for row in parameter_states]).reshape(N_axis, N_axis)
state_labels = [f'{state:03b}' for state in range(8)]
state_cmap = colors.ListedColormap(plt.get_cmap('tab10').colors[:8])
state_norm = colors.BoundaryNorm(jnp.arange(-0.5, 8.5), state_cmap.N)

fig, ax = plt.subplots(figsize=(7, 5))
state_map = ax.pcolormesh(f_values_hz, A_values_mm, state_number_grid,
                          cmap=state_cmap, norm=state_norm, shading='nearest')
colorbar = fig.colorbar(state_map, ax=ax, ticks=range(8), label='Final state')
colorbar.ax.set_yticklabels(state_labels)
ax.set(xlabel='Frequency (Hz)', ylabel='Amplitude (mm)',
       title='Final state after one sine-cycle input')
plt.tight_layout()